In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# clone autopacmen repository and enter path to it
# this serves to reuse autopacmns functions
import sys
sys.path.append('C:/Users/emanuel.lange/git/autopacmen')

# Creation of uncalibrated sMOMENT models

This script executes all sMOMENT functions to incorporate proteomics data into the initial model.

In [3]:
import cobra
import pandas
import json
import shutil

# read dataset assignment; every row represent sone donor/condition pairing for which a model should be generated
from autopacmen.submodules.get_initial_spreadsheets import get_initial_spreadsheets_with_sbml
from autopacmen.submodules.create_smoment_model_reaction_wise import create_smoment_model_reaction_wise

from utilities.utils import get_original_reaction_id

# Paths

In [4]:
parameterized_models_path_dict = {
    'Mitocore_Original': './../parameterized_plt_models/Mitocore_Original_plt.xml',
    'Mitocore_Preliminary': './../parameterized_plt_models/Mitocore_Preliminary_plt.xml',
    'Mitocore_MitoMammal': './../parameterized_plt_models/Mitocore_MitoMammal_plt.xml',
    'Mitocore_aligned_to_Human1': './../parameterized_plt_models/Mitocore_aligned_to_Human1_plt.xml',
}

smoment_paths_dict = {}

for model_name in parameterized_models_path_dict.keys():
    if model_name == "Mitocore_Original":
        continue
    smoment_paths_dict[model_name] = {
        "kcat_mapping_path": f"./../public_information/{model_name}_reactions_kcat_mapping_combined.json",
        "smoment_model_path": f"./../autopacmen_output/{model_name}/",
        "protein_file_output": f'./../model_data_input/{model_name}_protein_data.xlsx',
        "combined_database_path": f"./../public_information/{model_name}_combined_database.json",
        "mol_masss_mapping": f"./../model_data_input/{model_name}_protein_id_mass_mapping.json",
    }

In [5]:
cobra_models = {}

for model_name in parameterized_models_path_dict.keys():
    cobra_models[model_name] = cobra.io.read_sbml_model(parameterized_models_path_dict[model_name])

In [14]:
seahorse_df = pandas.read_excel('./../model_data_input/seahorse_data.xlsx')
max_atp = seahorse_df.loc[seahorse_df['parameter'] == 'ATP synthesis rate max', 'value_mmol_gdw_h'].values[0]

## 1) Get spreadsheets for data input and create uncalibrated models

In [15]:
def get_reaction_without_kcat(reaction_kcat_mapping_path, model):

    # get reactions that dont have kcat
    with open(reaction_kcat_mapping_path) as file:
        kcat_dict = json.load(file)

    exclude_reactions = []

    for reaction_id, kcats in kcat_dict.items():
        if str(kcats['forward']) == 'nan':
            exclude_reactions.append(reaction_id)

    # for some reactions no kcat is available, exclude those
    model = cobra.io.read_sbml_model(model)

    for reaction in model.reactions:
        if reaction.id not in kcat_dict.keys():
            exclude_reactions.append(reaction.id)

    return exclude_reactions

In [16]:

# reactions that do not have a found kcat should not receive a protein constraint.
# if not excluded the median over all kcat values will be introduced. This may cause some reactions to become rate-limiting.
# The maximal factor an initial kcat is allowed to be changed is 100 and together with the selected median kcats we actually 
# don't know for sure if the resulting kcat (after calibration) is anywehere close to the real one and would be rate limiting in reality.
# So better omit those :)
excluded_reactions = {}

for model_name in parameterized_models_path_dict.keys():
    if model_name == "Mitocore_Original":
        continue
    excluded_reactions[model_name] = get_reaction_without_kcat(smoment_paths_dict[model_name]["kcat_mapping_path"], parameterized_models_path_dict[model_name])

In [18]:
# only do if "urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed"
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [17]:
def perform_smoment(test_fba_dict, model_path, output_dir, project_name, protein_xlsx, model_specific_combined_database_path, reaction_kcat_mapping_path, molweight_mapping, exclude_reactions):

    model = cobra.io.read_sbml_model(model_path)
    
    for metabolite in model.metabolites:
        if metabolite.formula == 'N/A':
            metabolite.formula = None
            print(metabolite.id)

    # generate spreadsheets
    get_initial_spreadsheets_with_sbml(model_path, output_dir, project_name)

    # copy protein data file to model directory
    shutil.copy2(protein_xlsx, output_dir)

    # copy mol masses json and model specific json files to model directory
    shutil.copy2(molweight_mapping, output_dir)
    shutil.copy2(model_specific_combined_database_path, output_dir)

    shutil.copy2(reaction_kcat_mapping_path, str(output_dir) + '/' + project_name + '_reactions_kcat_mapping_combined.json')

    # create uncalibrated model
    with model:
        create_smoment_model_reaction_wise(model, project_name + '_uncalibrated.xml', output_dir, project_name, exclude_reactions, "median")

    uncalibrated_model = cobra.io.read_sbml_model(str(output_dir) + '/' + project_name + '_uncalibrated.xml')

    pfba_solution_uncalibrated = cobra.flux_analysis.pfba(uncalibrated_model)
    test_fba_dict['model'].append(project_name)
    test_fba_dict['fba_value'].append(pfba_solution_uncalibrated.fluxes['OF_ATP_MitoCore'])
    test_fba_dict['seahorse_flux'].append(max_atp)

In [19]:

solution = cobra.flux_analysis.pfba(cobra_models['Mitocore_Original'])
test_fba_dict = {'model': ['mitocore'], 'fba_value': [solution.fluxes['OF_ATP_MitoCore']], 'seahorse_flux': [max_atp]}

for model_name in parameterized_models_path_dict.keys():
    if model_name == "Mitocore_Original":
        continue
    
    print(model_name)
    perform_smoment(
        test_fba_dict,
        parameterized_models_path_dict[model_name],
        smoment_paths_dict[model_name]["smoment_model_path"],
        model_name,
        smoment_paths_dict[model_name]["protein_file_output"],
        smoment_paths_dict[model_name]["combined_database_path"],
        smoment_paths_dict[model_name]["kcat_mapping_path"],
        smoment_paths_dict[model_name]["mol_masss_mapping"],
        excluded_reactions[model_name]
    )

pandas.DataFrame(test_fba_dict)

Mitocore_Preliminary
biomass_c
biomass_e
biomass_m
INFO: Reaction EX_2hb_e does not have a KEGG ID annotation
INFO: Reaction EX_ac_e does not have a KEGG ID annotation
INFO: Reaction EX_acac_e does not have a KEGG ID annotation
INFO: Reaction EX_akg_e does not have a KEGG ID annotation
INFO: Reaction EX_ala_B_e does not have a KEGG ID annotation
INFO: Reaction EX_ala_L_e does not have a KEGG ID annotation
INFO: Reaction EX_arg_L_e does not have a KEGG ID annotation
INFO: Reaction EX_argsuc_e does not have a KEGG ID annotation
INFO: Reaction EX_asn_L_e does not have a KEGG ID annotation
INFO: Reaction EX_asp_L_e does not have a KEGG ID annotation
INFO: Reaction EX_bhb_e does not have a KEGG ID annotation
INFO: Reaction EX_bilirub_e does not have a KEGG ID annotation
INFO: Reaction EX_biomass_e does not have a KEGG ID annotation
INFO: Reaction EX_but_e does not have a KEGG ID annotation
INFO: Reaction EX_chol_e does not have a KEGG ID annotation
INFO: Reaction EX_cit_e does not have a KE

c:\Users\emanuel.lange\.conda\envs\smoment\lib\site-packages\cobra\core\group.py:107: UserWarning: need to pass in a list
  warn("need to pass in a list")


splitting CSm
splitting ACONTm
splitting ICDHxm
splitting ICDHyrm
splitting AKGDm
splitting SUCOAS1m
splitting SUCOASm
splitting FUMm
splitting MDHm
splitting CI_MitoCore
splitting CII_MitoCore
splitting CIII_MitoCore
splitting CIV_MitoCore
splitting CV_MitoCore
splitting PEPCKm
splitting PCm
splitting ME2m
splitting ME1m
splitting r0081
splitting ACITLm_MitoCore
splitting NDPK1m
splitting NNT_MitoCore
splitting ADK1m
splitting ME2
splitting ALATA_L
splitting NDPK1
splitting FUM
splitting ADK1
splitting ICDHy
splitting ACONT
splitting ACITL
splitting ASPTA
splitting MDH
splitting AKGMALtm
splitting ASPGLUmB_MitoCore
splitting ASPTAm
splitting G3PD1
splitting r0205
splitting FACOAL160i
splitting C160CPT1
splitting PPA
splitting r2435
splitting C160CPT2
splitting PPAm
splitting ACOT2_MitoCore
splitting ACADLC16_MitoCore
splitting MECR16C_MitoCore
splitting MTPC16_MitoCore
splitting ACADLC14_MitoCore
splitting MECR14C_MitoCore
splitting MTPC14_MitoCore
splitting r1447
splitting r0638
spli

,model,fba_value,seahorse_flux
0,mitocore,6.521542,2.194366
1,Mitocore_Preliminary,0.264572,2.194366
2,Mitocore_MitoMammal,0.460253,2.194366
3,Mitocore_aligned_to_Human1,0.497266,2.194366


In [29]:
def check_orphan_deliveries(model):
    
    orphan_deliveries = []
    
    for reaction in model.reactions:
        if reaction.id.startswith('ENZYME_DELIVERY_'):
            metabolites = list(reaction.metabolites.keys())
            if len(metabolites) > 1:
                continue
            metabolite = metabolites[0]
            
            if len(metabolite.reactions) == 1:
                orphan_deliveries.append(reaction.id)
    
    print(f'model contains {len(orphan_deliveries)} orphan deliveries.')
    print(orphan_deliveries)
    
    return orphan_deliveries

In [32]:
for model_name in parameterized_models_path_dict.keys():
    if model_name == "Mitocore_Original":
        continue
    
    print(model_name)
    # smoment models contain nzyme delivery reaction that generate enzymes which are not conncted to any other reaction.
    # Seems like those interfere with the flux sampling with hopsy and are deleted therefore.
    uncalibratd_model_path = smoment_paths_dict[model_name]["smoment_model_path"] + model_name + '_uncalibrated.xml'
    
    uncalibrated_model = cobra.io.read_sbml_model(uncalibratd_model_path)
    print(uncalibrated_model.optimize())
    
    orphan_deliveries = check_orphan_deliveries(uncalibrated_model)
    
    uncalibrated_model.remove_reactions(orphan_deliveries, remove_orphans=True)
    print(uncalibrated_model.optimize())
    
    check_orphan_deliveries(uncalibrated_model)
    
    cobra.io.write_sbml_model(uncalibrated_model, uncalibratd_model_path)

Mitocore_Preliminary
<Solution 0.265 at 0x2157e702110>
model contains 19 orphan deliveries.
['ENZYME_DELIVERY_ENSG00000178537', 'ENZYME_DELIVERY_ENSG00000115840', 'ENZYME_DELIVERY_ENSG00000143158', 'ENZYME_DELIVERY_ENSG00000005022', 'ENZYME_DELIVERY_ENSG00000169100', 'ENZYME_DELIVERY_ENSG00000171314', 'ENZYME_DELIVERY_ENSG00000066926', 'ENZYME_DELIVERY_ENSG00000157184', 'ENZYME_DELIVERY_ENSG00000100075', 'ENZYME_DELIVERY_ENSG00000131473', 'ENZYME_DELIVERY_ENSG00000075415', 'ENZYME_DELIVERY_ENSG00000108528', 'ENZYME_DELIVERY_ENSG00000093144', 'ENZYME_DELIVERY_ENSG00000112697', 'ENZYME_DELIVERY_ENSG00000183048', 'ENZYME_DELIVERY_ENSG00000004864', 'ENZYME_DELIVERY_ENSG00000058063', 'ENZYME_DELIVERY_ENSG00000124406', 'ENZYME_DELIVERY_ENSG00000102743']
<Solution 0.265 at 0x21502d37d30>
model contains 0 orphan deliveries.
[]
Mitocore_MitoMammal
<Solution 0.460 at 0x2157d693580>
model contains 52 orphan deliveries.
['ENZYME_DELIVERY_ENSG00000110717', 'ENZYME_DELIVERY_ENSG00000189043', 'ENZYME

## 2) Determine candidates for calibration

In [23]:
from utilities.kcat_sensitivity import get_kcat_candidates

In [24]:

smoment_models = {}
kcat_candidates_dict = {}

for model_name in parameterized_models_path_dict.keys():
    if model_name == "Mitocore_Original":
        continue
    smoment_models[model_name] = cobra.io.read_sbml_model(smoment_paths_dict[model_name]["smoment_model_path"] + '/' + model_name + '_uncalibrated.xml')

    print(f' {model_name} model sensitivity analysis coming up...')
    kcat_candidates, max_change_factor =\
        get_kcat_candidates(smoment_models[model_name], 'OF_ATP_MitoCore', 5, max_atp, 0, 0.2, 0.001, do_all_iterations=True)
            
    kcat_candidates_dict[model_name] = {
        "reactions":  kcat_candidates.set_index('matlab_reaction_id').to_dict('index'),
        "max_change_factor": max_change_factor
    }
    
with open('./../autopacmen_output/kcat_candidates.json', 'w') as f:
    json.dump(kcat_candidates_dict, f)

 Mitocore_Preliminary model sensitivity analysis coming up...
orignal objective: 0.2645718714519445
Checking sensitivity in iteration 0


c:\Users\emanuel.lange\git\mitocore-smoment-model\smoment\scripts\utilities\kcat_sensitivity.py:139: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  kcat_sensitivity = pandas.concat([kcat_sensitivity, new_row], ignore_index=True)


Scaling kcats in reaction: LDH_L_GPRSPLIT_1_TG_reverse by 10 resulting objective:  0.46548712136099696
Scaling kcats in reaction: CV_MitoCore_GPRSPLIT_1_TG_forward by 10 resulting objective:  0.5018452151331374
Scaling kcats in reaction: CV_MitoCore_GPRSPLIT_2_TG_forward by 10 resulting objective:  0.5018452151331371
Scaling kcats in reaction: CV_MitoCore_GPRSPLIT_3_TG_forward by 10 resulting objective:  0.5018452151331374
Scaling kcats in reaction: CIII_MitoCore_GPRSPLIT_1_TG_forward by 10 resulting objective:  0.8604396411377765
Checking sensitivity in iteration 1


c:\Users\emanuel.lange\git\mitocore-smoment-model\smoment\scripts\utilities\kcat_sensitivity.py:139: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  kcat_sensitivity = pandas.concat([kcat_sensitivity, new_row], ignore_index=True)


Scaling kcats in reaction: CIV_MitoCore_GPRSPLIT_1 by 10 resulting objective:  1.19896641552507
Scaling kcats in reaction: CI_MitoCore_GPRSPLIT_1_TG_forward by 10 resulting objective:  1.5470380997505855
Checking sensitivity in iteration 2


...

Checking sensitivity in iteration 3


c:\Users\emanuel.lange\git\mitocore-smoment-model\smoment\scripts\utilities\kcat_sensitivity.py:139: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  kcat_sensitivity = pandas.concat([kcat_sensitivity, new_row], ignore_index=True)


Checking sensitivity in iteration 4


c:\Users\emanuel.lange\git\mitocore-smoment-model\smoment\scripts\utilities\kcat_sensitivity.py:139: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  kcat_sensitivity = pandas.concat([kcat_sensitivity, new_row], ignore_index=True)


max change factor: 10


# Included constraints

In [33]:
def get_ec_statistics(model: cobra.Model):
    
    # get number of enzyme reactions
    enzyme_reaction_counter = 0
    split_reaction_counter = 0
    original_reaction_with_ec_counter = set()

    for reaction in model.reactions:
        if reaction.id.startswith("ENZYME_DELIVERY_"):
            enzyme_reaction_counter += 1
        if "_GPRSPLIT_" in reaction.id:
            split_reaction_counter += 1
        if "_TG_" in reaction.id:
            original_id, reverse, isoenzyme_split_index = get_original_reaction_id(reaction.id)
            original_reaction_with_ec_counter.add(original_id)
    
    print(f"Found {enzyme_reaction_counter} enzyme delivery reactions.")
    print(f"Found {split_reaction_counter} split reactions.")
    print(f"Found {len(original_reaction_with_ec_counter)} original reactions with EC number that received a protein constraint.")

In [34]:
for model_name in smoment_paths_dict.keys():
    print(f"{model_name} EC number statistics:")
    model = cobra.io.read_sbml_model(smoment_paths_dict[model_name]['smoment_model_path'] + model_name + '_uncalibrated.xml')
    get_ec_statistics(model)
    check_orphan_deliveries(model)

Mitocore_Preliminary EC number statistics:
Found 209 enzyme delivery reactions.
Found 376 split reactions.
Found 168 original reactions with EC number that received a protein constraint.
model contains 0 orphan deliveries.
[]
Mitocore_MitoMammal EC number statistics:
Found 176 enzyme delivery reactions.
Found 426 split reactions.
Found 170 original reactions with EC number that received a protein constraint.
model contains 0 orphan deliveries.
[]
Mitocore_aligned_to_Human1 EC number statistics:
Found 234 enzyme delivery reactions.
Found 750 split reactions.
Found 202 original reactions with EC number that received a protein constraint.
model contains 0 orphan deliveries.
[]
